In [1]:
#r "nuget: Microsoft.ML.OnnxRuntime"
#r "nuget: Microsoft.ML"
#r "nuget: Microsoft.Data.Analysis"

Installed Packages Microsoft.Data.Analysis, 0.23.0 Microsoft.ML, 5.0.0 Microsoft.ML.OnnxRuntime, 1.23.2

Loading extensions from `/home/jrhol/.nuget/packages/microsoft.data.analysis/0.23.0/interactive-extensions/dotnet/Microsoft.Data.Analysis.Interactive.dll`

In [2]:
using Microsoft.ML.OnnxRuntime;
using Microsoft.ML.OnnxRuntime.Tensors;
using Microsoft.ML.Data;
using Microsoft.Data.Analysis;
using System.IO;
using System.Linq;
using System.Globalization;

In [3]:
var modelPath = "../models/fraud_xgb_classifier.onnx";
var dataPath = "../data/processed/creditcard.parquet";

In [4]:
public static class Metrics
{
    public static double Accuracy(IEnumerable<int> actualLabel, IEnumerable<int> predictedLabel)
    {
        return actualLabel.Zip(predictedLabel, (t, p) => t == p ? 1 : 0).Average();
    }

    public static double Precision(IEnumerable<int> actualLabel, IEnumerable<int> predictedLabel)
    {
        var tp = actualLabel.Zip(predictedLabel, (t, p) => t == 1 && p == 1 ? 1 : 0).Sum();
        var fp = actualLabel.Zip(predictedLabel, (t, p) => t == 0 && p == 1 ? 1 : 0).Sum();
        return tp / (double)(tp + fp);
    }

    public static double Recall(IEnumerable<int> actualLabel, IEnumerable<int> predictedLabel)
    {
        var tp = actualLabel.Zip(predictedLabel, (t, p) => t == 1 && p == 1 ? 1 : 0).Sum();
        var fn = actualLabel.Zip(predictedLabel, (t, p) => t == 1 && p == 0 ? 1 : 0).Sum();
        return tp / (double)(tp + fn);
    }

    public static double F1Score(IEnumerable<int> actualLabel, IEnumerable<int> predictedLabel)
    {
        var precision = Precision(actualLabel, predictedLabel);
        var recall = Recall(actualLabel, predictedLabel);
        return 2 * (precision * recall) / (precision + recall);
    }
}

In [5]:
public static (double, double) ToCyclicFeature(int elapsedTime)
{
    double period = 24 * 60 * 60; // seconds in a day
    double angle = 2 * Math.PI * elapsedTime / period;
    return (Math.Sin(angle), Math.Cos(angle));
}

public static IEnumerable<DenseTensor<float>> TensorBatchesFromDataFrame(DataFrame df, int batchSize)
{
    var size = Math.Max(1, batchSize);
    var batch = new List<float[]>(size);
    
    foreach (var dfRow in df.Rows)
    {
        var row = new float[df.Columns.Count];
        for (int i = 0; i < df.Columns.Count; i++)
        {
            row[i] = Convert.ToSingle(dfRow[i]);
        }
        batch.Add(row);

        if (batch.Count == size)
        {
            yield return ToTensor(batch);
            batch.Clear();
        }
    }
}

public static DenseTensor<float> ToTensor(IList<float[]> batch)
{
    if (batch is null || batch.Count == 0) throw new ArgumentException("Batch is empty");

    int batchSize = batch.Count;
    int featureCount = batch[0].Length;
    var tensor = new DenseTensor<float>(new[] { batchSize, featureCount });

    for (int i = 0; i < batchSize; i++)
    {
        for (int j = 0; j < featureCount; j++)
        {
            tensor[i, j] = batch[i][j];
        }
    }

    return tensor;
}

In [6]:
List<string> columns = ["v1", "v2", "v3", "v4", "v5", "v6", "v7", "v8", "v9", "v10", "v11", "v12", "v13", "v14", "v15", "v16", "v17", "v18", "v19", "v20", "v21", "v22", "v23", "v24", "v25", "v26", "v27", "v28", "amount", "hour_sin", "hour_cos", "is_fraud"];

In [7]:
DataFrame df = DataFrame.LoadCsv("../data/processed/test.csv"); 

In [8]:
DataFrame featureMatrix = new DataFrame(columns.Select(name => df[name]));
featureMatrix.Columns.Remove("is_fraud");

var fraudColumn = ((SingleDataFrameColumn) df["is_fraud"]).Select(v => v.HasValue && v.Value != 0f ? 1 : 0);;


In [9]:
var stream = TensorBatchesFromDataFrame(featureMatrix, 512);

In [10]:
var session = new InferenceSession(modelPath);

In [11]:
var inputs = session.InputMetadata.Keys.ToList();
Console.WriteLine($"Model input inputs: {string.Join(", ", inputs)}");

var outputs = session.OutputMetadata.Keys.ToList();
Console.WriteLine($"Model output outputs: {string.Join(", ", outputs)}");

Model input inputs: input
Model output outputs: output_label, output_probability


In [ ]:
List<int> preds = [];

foreach (var inputBatch in TensorBatchesFromDataFrame(featureMatrix, 512))
{
    var results = session.Run(new[] { NamedOnnxValue.CreateFromTensor("input", inputBatch) });
    var t = results.First().AsTensor<Int64>();
    preds.AddRange(t.Select(v => (int) v).ToArray());
}

Error: Microsoft.ML.OnnxRuntime.OnnxRuntimeException: [ErrorCode:InvalidArgument] Input name: 'features' is not in the metadata
   at Microsoft.ML.OnnxRuntime.InferenceSession.LookupInputMetadata(String nodeName) in E:\_work\1\s\csharp\src\Microsoft.ML.OnnxRuntime\InferenceSession.shared.cs:line 938
   at Microsoft.ML.OnnxRuntime.InferenceSession.LookupUtf8Names[T](IReadOnlyCollection`1 values, NameExtractor`1 nameExtractor, MetadataLookup metaLookup) in E:\_work\1\s\csharp\src\Microsoft.ML.OnnxRuntime\InferenceSession.shared.cs:line 996
   at Microsoft.ML.OnnxRuntime.InferenceSession.Run(IReadOnlyCollection`1 inputs, IReadOnlyCollection`1 outputNames, RunOptions options) in E:\_work\1\s\csharp\src\Microsoft.ML.OnnxRuntime\InferenceSession.shared.cs:line 326
   at Microsoft.ML.OnnxRuntime.InferenceSession.Run(IReadOnlyCollection`1 inputs, IReadOnlyCollection`1 outputNames) in E:\_work\1\s\csharp\src\Microsoft.ML.OnnxRuntime\InferenceSession.shared.cs:line 312
   at Microsoft.ML.OnnxRun

In [13]:
Console.WriteLine($"Total predictions: {preds.Count}");
Console.WriteLine($"Accuracy: {Metrics.Accuracy(fraudColumn, preds)}");
Console.WriteLine($"Precision: {Metrics.Precision(fraudColumn, preds)}");
Console.WriteLine($"Recall: {Metrics.Recall(fraudColumn, preds)}");
Console.WriteLine($"F1 Score: {Metrics.F1Score(fraudColumn, preds)}");

Total predictions: 0


Error: System.InvalidOperationException: Sequence contains no elements
   at System.Linq.ThrowHelper.ThrowNoElementsException()
   at System.Linq.Enumerable.Average(IEnumerable`1 source)
   at Submission#4.Metrics.Accuracy(IEnumerable`1 actualLabel, IEnumerable`1 predictedLabel)
   at Submission#13.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)